In [1]:
# Cell 1
!pip install torch --index-url https://download.pytorch.org/whl/cu124 -q
!pip install --no-deps "xformers<0.0.29" "trl<0.9.0" peft accelerate bitsandbytes -q
!pip install --no-deps unsloth unsloth_zoo -q
!pip install fastapi uvicorn nest_asyncio -q
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared
!chmod +x cloudflared

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.7/16.7 MB 84.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 245.2/245.2 kB 17.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 44.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.8/55.8 kB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.6/62.6 MB 45.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 418.4/418.4 kB 31.8 MB/s eta 0:00:00


In [3]:
# Cell 2: upload api_server.py
from google.colab import files
files.upload()  # api_server.py

import subprocess, threading

def run_server():
    subprocess.run(['python', 'api_server.py'])

t = threading.Thread(target=run_server)
t.daemon = True
t.start()

import time
time.sleep(5)
print('FastAPI server starting... (waitting)')

KeyboardInterrupt: 

In [ ]:
# Cell 3
import requests, time

for i in range(20):
    try:
        r = requests.get('http://localhost:8000/health')
        if r.json().get('status') == 'ok':
            print('FastAPI server ready ✅')
            break
    except:
        print(f'waitting... ({i+1}/20)')
        time.sleep(15)

In [ ]:
# Cell 4: start Cloudflare Tunnel（keep going）
# ptint address to code.html 7th. line
import subprocess, threading, time, re

def run_tunnel():
    proc = subprocess.Popen(
        ['./cloudflared', 'tunnel', '--url', 'http://localhost:8000'],
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT
    )
    for line in proc.stdout:
        line = line.decode().strip()
        urls = re.findall(r'https://[\w\-]+\.trycloudflare\.com', line)
        if urls:
            print(f'\naddress（fufill code.html 7th. line）: {urls[0]}/analyze\n')

threading.Thread(target=run_tunnel, daemon=True).start()
while True: time.sleep(60)

In [ ]:
# Cell 5: test（optional）
import requests, json

sample_prompt = """[USER GROUP: Adult/Diabetes Risk]
Age: 45 | Gender: Male | Weight: 91.7kg | Height: 174.8cm | BMI: 30.0
Waist: 106cm | WHR: 0.61 | Sleep: 7hrs
Conditions: Diabetes

Current Intake:
  Total:     2600cal | P:80g | F:110g | C:310g

Goal: Weight Maintenance
Request: Critique my current intake and recommend adjustments only."""

response = requests.post(
    'http://localhost:8000/analyze',
    headers={'Content-Type': 'application/json'},
    data=json.dumps({'prompt': sample_prompt})
)
print(response.json()['advice'])

ConnectionError: HTTPConnectionPool(host='localhost', port=8000): Max retries exceeded with url: /analyze (Caused by NewConnectionError('<urllib3.connection.HTTPConnection object at 0x7cae40f16e10>: Failed to establish a new connection: [Errno 111] Connection refused'))